Conectar seu Jupyter Notebook ao banco de dados PostgreSQL

In [8]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv("../.env")  # o notebook está em notebooks/, o .env na raiz

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

# Teste de conexão
with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


Organizar e mapear os arquivos CSV do dataset Olist antes de importá-los para o PostgreSQL.

In [9]:
RAW = Path("../data/raw")

# arquivo CSV: (nome da tabela no banco, colunas de data)
arquivos = {
    "olist_customers_dataset.csv":        ("customers", []),
    "olist_geolocation_dataset.csv":      ("geolocation", []),
    "olist_order_items_dataset.csv":      ("order_items", ["shipping_limit_date"]),
    "olist_order_payments_dataset.csv":   ("order_payments", []),
    "olist_order_reviews_dataset.csv":    ("order_reviews",
                                           ["review_creation_date", "review_answer_timestamp"]),
    "olist_orders_dataset.csv":           ("orders",
                                           ["order_purchase_timestamp", "order_approved_at",
                                            "order_delivered_carrier_date",
                                            "order_delivered_customer_date",
                                            "order_estimated_delivery_date"]),
    "olist_products_dataset.csv":         ("products", []),
    "olist_sellers_dataset.csv":          ("sellers", []),
    "product_category_name_translation.csv": ("product_category_translation", []),
}

Etapa de importação dos dados do Olist para o PostgreSQL

In [10]:
shapes_csv = {}

for arquivo, (tabela, colunas_data) in arquivos.items():
    df = pd.read_csv(RAW / arquivo, parse_dates=colunas_data)
    shapes_csv[tabela] = df.shape
    print(f"{tabela}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

    df.to_sql(
        name=tabela,
        con=engine,
        if_exists="replace",   # recria a tabela se já existir
        index=False,           # não grava o índice do pandas como coluna
        chunksize=10_000,      # grava em lotes (a geolocation tem ~1 milhão de linhas)
    )

customers: 99,441 linhas x 5 colunas
geolocation: 1,000,163 linhas x 5 colunas
order_items: 112,650 linhas x 7 colunas
order_payments: 103,886 linhas x 5 colunas
order_reviews: 99,224 linhas x 7 colunas
orders: 99,441 linhas x 8 colunas
products: 32,951 linhas x 9 colunas
sellers: 3,095 linhas x 4 colunas
product_category_translation: 71 linhas x 2 colunas


Validar se todos os dados dos arquivos CSV foram importados corretamente para o PostgreSQL.

In [11]:
with engine.connect() as conn:
    for tabela, (linhas, colunas) in shapes_csv.items():
        no_banco = conn.execute(text(f"SELECT COUNT(*) FROM {tabela}")).scalar()
        status = "OK" if no_banco == linhas else "DIVERGENTE"
        print(f"{tabela}: CSV={linhas:,} | banco={no_banco:,} -> {status}")

customers: CSV=99,441 | banco=99,441 -> OK
geolocation: CSV=1,000,163 | banco=1,000,163 -> OK
order_items: CSV=112,650 | banco=112,650 -> OK
order_payments: CSV=103,886 | banco=103,886 -> OK
order_reviews: CSV=99,224 | banco=99,224 -> OK
orders: CSV=99,441 | banco=99,441 -> OK
products: CSV=32,951 | banco=32,951 -> OK
sellers: CSV=3,095 | banco=3,095 -> OK
product_category_translation: CSV=71 | banco=71 -> OK
